# Delivery Time Prediction with TabPFN-3.5 - v3 (contexto de 200k e 300k na T4)

O v2 mostrou que a curva de aprendizado do TabPFN ainda não estabilizou em 100k linhas
(MAE 5,073 no teste completo; cerca de -0,07 min a cada vez que o contexto dobra).
O v3 tenta **200.000** e **300.000** linhas de contexto numa **T4 (15 GB)**.

### Por que não basta aumentar `sample_size`
- No v2, 100k linhas usaram **8,7 GB** de pico. O KV-cache do `fit_with_cache` cresce de forma
  linear com o contexto e com o número de estimadores internos (`n_estimators`):
  200k com o mesmo `n_estimators` fica no limite da T4, e 300k passa dele.
- `fit_preprocessors` (sem cache) não serve aqui: ele recalcula o contexto inteiro a cada lote do teste,
  e o fit de 100k já levou ~15 min.

### O que o v3 faz
1. **Sonda de memória (Stage 3):** mede a VRAM de pico em contextos pequenos (10k e 30k) para cada
   `n_estimators` candidato (`auto`, 4, 2, 1), extrapola linearmente para 200k/300k (calibrando com
   os 8,7 GB medidos no v2) e escolhe o **maior `n_estimators` que cabe**.
2. **Contexto único de 200k e 300k (Stage 4):** se o fit der OOM mesmo assim, tenta o próximo
   `n_estimators` menor. OOM no predict só reduz o lote, sem refazer o fit.
3. **Fallback garantido para 300k (Stage 5):** se 300k não couber em contexto único, roda
   **3 contextos disjuntos de 100k** (a configuração do v2, que sabidamente cabe) e faz a média.
   Assim, as 300k linhas são usadas de qualquer forma.
4. **Comparação com o v2:** mesmas métricas, mesmo subconjunto de teste da curva de aprendizado,
   LightGBM com a mesma amostra e blend com peso escolhido na validação.

**Tempo esperado:** de 1,5 a 3 horas na T4 (a sonda imprime uma estimativa antes de começar).
Mantenha a aba aberta e, se puder, preencha `DRIVE_BACKUP_DIR` no Stage 0 para salvar os
resultados a cada cenário.

## Stage 0: Colab Setup, Authentication, and Project Bundle

1. **Runtime -> Change runtime type -> T4 GPU**.
2. `TABPFN_TOKEN` nos Colab Secrets.
3. Envie o `meituan_colab_bundle.zip` quando for pedido (gerado com `python pack_colab_bundle.py`).

In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
# Reduz fragmentação da VRAM (precisa ser definido ANTES de importar torch)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
WORK_ROOT = Path('/content/meituan-project')
DRIVE_PROJECT_DIR = ''  # Optional: set to '/content/drive/MyDrive/...' if using Google Drive
# Opcional: pasta no Google Drive para salvar os resultados após cada cenário
# (protege contra desconexão do Colab em execuções longas). Ex.: '/content/drive/MyDrive/tabpfn_v3'
DRIVE_BACKUP_DIR = ''

# If dataloader was already imported in Python memory from an earlier run, clear it so the new one loads
for mod in ['dataloader', 'tabpfn_model']:
    if mod in sys.modules:
        del sys.modules[mod]

if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tabpfn==10.2.0', 'pyarrow', 'lightgbm'], check=True)

    if DRIVE_PROJECT_DIR and Path(DRIVE_PROJECT_DIR).exists():
        WORK_ROOT = Path(DRIVE_PROJECT_DIR)
    else:
        WORK_ROOT.mkdir(parents=True, exist_ok=True)
        local_zip = Path('/content/meituan_colab_bundle.zip')

        # Check if dataloader already exists and has the new features
        dataloader_file = WORK_ROOT / 'data' / 'dataloader.py'
        need_extract = True
        if dataloader_file.exists():
            content = dataloader_file.read_text(encoding='utf-8')
            if 'EXTRA_REDUCED_COLS' in content:
                print('dataloader.py atualizado já presente no disco.')
                need_extract = False

        if need_extract:
            if not local_zip.exists():
                from google.colab import files
                print('Por favor, selecione meituan_colab_bundle.zip atualizado para upload:')
                uploaded = files.upload()
                zip_names = [k for k in uploaded if k.endswith('.zip')]
                if not zip_names:
                    raise RuntimeError('Nenhum arquivo .zip foi enviado.')
                local_zip = Path('/content') / zip_names[0]

            print(f'Extraindo {local_zip.name} para {WORK_ROOT} (sobrescrevendo arquivos antigos)...')
            with zipfile.ZipFile(local_zip, 'r') as zf:
                zf.extractall(WORK_ROOT)

        # Sanity check: verify dataloader.py has the new features
        dl_text = (WORK_ROOT / 'data' / 'dataloader.py').read_text(encoding='utf-8')
        if 'EXTRA_REDUCED_COLS' not in dl_text:
            raise RuntimeError(
                "ATENÇÃO: O dataloader.py extraído é a versão ANTIGA (sem EXTRA_REDUCED_COLS).\n"
                "Solução: No seu computador, rode 'python pack_colab_bundle.py'.\n"
                "Em seguida, no Colab rode: '!rm -rf /content/meituan_colab_bundle.zip /content/meituan-project' e execute esta célula de novo enviando o novo zip."
            )

    nb_dir = WORK_ROOT / 'notebooks_analysis'
    nb_dir.mkdir(parents=True, exist_ok=True)
    os.chdir(nb_dir)

    if not os.environ.get('TABPFN_TOKEN'):
        try:
            from google.colab import userdata
            token = userdata.get('TABPFN_TOKEN')
            if token:
                os.environ['TABPFN_TOKEN'] = token.strip()
        except Exception:
            pass

    if DRIVE_BACKUP_DIR:
        from google.colab import drive
        drive.mount('/content/drive')
        Path(DRIVE_BACKUP_DIR).mkdir(parents=True, exist_ok=True)

    if not os.environ.get('TABPFN_TOKEN'):
        raise RuntimeError('TABPFN_TOKEN not found. Add it to Colab Secrets.')
    print('Colab setup complete. Working dir:', Path.cwd())
else:
    print('Running outside Colab. Working dir:', Path.cwd())

## Stage 1: Configuração

In [ ]:
import gc
import json
import math
import shutil
import time
import zipfile

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tabpfn
import torch

sys.path += ['../data', '../models']
from dataloader import BASELINE, TARGET, FEATURES_COMPLETE_EXTRA, load_meituan
from tabpfn_model import TabPFNConfig, TabPFNModel, compute_metrics, sample_dataframe

if not torch.cuda.is_available():
    raise RuntimeError('O v3 precisa de GPU (contextos de 200k/300k). Selecione um runtime com T4.')

SEED = 42
CKPT_DIR = Path('../models/checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_SET_NAME = 'FEATURES_COMPLETE_EXTRA'
FEAT_COLS = FEATURES_COMPLETE_EXTRA          # 38 features, iguais ao v2
CATEGORICAL_COLS = ('district_id',)
OUTPUT_TYPE = 'median'                       # mediana minimiza MAE
COMPUTE_MEAN = True                          # média: melhor RMSE/R² (custa mais uma passada no teste)
BATCH_SIZE = 2048

TARGET_SIZES = [200_000, 300_000]            # contextos únicos a tentar
N_EST_CANDIDATES = ['auto', 4, 2, 1]         # do mais preciso ao mais leve
FORCE_N_ESTIMATORS = {}                      # ex.: {300_000: 2} para pular a sonda nesse tamanho
PROBE_SIZES = (10_000, 30_000)
VRAM_SAFETY = 0.90                           # fração da VRAM livre que o pico previsto pode usar
V2_PEAK_100K_GB = 8.71                       # pico medido no v2 (100k, n_estimators='auto')
TIME_EXPONENT = 1.7                          # fit ~ n^1.7 (medido na curva do v2)

RUN_DISJOINT_ENSEMBLE = 'auto'               # 'auto' = só se 300k não couber; True = sempre; False = nunca
DISJOINT_MEMBER_SIZE = 100_000

VAL_BLEND_SIZE = 20_000                      # subconjunto da validação para escolher o peso do blend
LC_EVAL_SIZE = 10_000                        # mesmo subconjunto de teste da curva do v2

gpu_name = torch.cuda.get_device_name(0)
vram_total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name} ({vram_total_gb:.1f} GB) | tabpfn={tabpfn.__version__} | features={len(FEAT_COLS)}')


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()


def is_oom(exc):
    return isinstance(exc, torch.cuda.OutOfMemoryError) or 'out of memory' in str(exc).lower()


def make_cfg(n, n_est, verbose=False):
    return TabPFNConfig(
        model_version='v3.5', n_ensemble=1, sample_size=n, sampling_strategy='stratified',
        n_estimators=n_est, fit_mode='fit_with_cache', n_preprocessing_jobs=1,
        batch_size=BATCH_SIZE, aggregation='mean', output_type=OUTPUT_TYPE,
        categorical_cols=CATEGORICAL_COLS, ignore_pretraining_limits=True,
        seed=SEED, device='cuda', verbose=verbose,
    )


def predict_with_retry(model, X, output_type, min_batch=128):
    """Predict; on OOM halve the batch size WITHOUT refitting (the KV-cache stays on the GPU).

    Memory is freed outside the except block, where the traceback no longer pins tensors.
    """
    while True:
        oom = False
        try:
            return model.predict(X, output_type=output_type)
        except Exception as exc:
            if not is_oom(exc):
                raise
            oom = True
        if oom:
            free_gpu()
            new_bs = model.config.batch_size // 2
            if new_bs < min_batch:
                raise torch.cuda.OutOfMemoryError('predict não coube nem com lote mínimo')
            model.config.batch_size = new_bs
            print(f'  [OOM no predict] batch_size -> {new_bs}')


def resolved_n_estimators(model):
    """Best effort: how many internal estimators 'auto' resolved to (None if not exposed)."""
    est = model.estimators_[0]
    for attr in ('n_estimators_', 'n_estimators'):
        v = getattr(est, attr, None)
        if isinstance(v, (int, np.integer)):
            return int(v)
    for attr in ('ensemble_configs_', 'ensemble_configs'):
        v = getattr(est, attr, None)
        if v is not None:
            try:
                return len(v)
            except TypeError:
                pass
    return None


def backup(*paths):
    """Copy result files to Google Drive (if DRIVE_BACKUP_DIR is set in Stage 0)."""
    if not DRIVE_BACKUP_DIR:
        return
    for p in paths:
        p = Path(p)
        if p.exists():
            shutil.copy2(p, Path(DRIVE_BACKUP_DIR) / p.name)


def metric_row(name, n_train, m, runtime_s, **extra):
    return {'model': name, 'train_rows': n_train, 'test_MAE': round(m['MAE'], 3),
            'test_RMSE': round(m['RMSE'], 3), 'test_R2': round(m['R2'], 4),
            'runtime_s': round(runtime_s, 1), **extra}


plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## Stage 2: Dados, baseline do app e LightGBM com todo o treino

Mesmo split do v2 (`70_30`, Manhattan). O LightGBM com as 377k linhas é a referência e a
segunda perna do blend.

In [ ]:
t0 = time.time()
tr_70, val_70, te_70 = load_meituan(split='70_30', distance='manhattan', seed=SEED, verbose=True)
tr_70, val_70, te_70 = tr_70.copy(), val_70.copy(), te_70.copy()
for df in (tr_70, val_70, te_70):
    df['_y_bin'] = pd.qcut(df[TARGET], q=20, labels=False, duplicates='drop')

te_eval = te_70.reset_index(drop=True)          # teste completo (53.872 linhas)
y_te = te_eval[TARGET].to_numpy()
te_lc, lc_idx, _ = sample_dataframe(te_70, LC_EVAL_SIZE, stratify_col='_y_bin', seed=SEED)
y_lc = te_lc[TARGET].to_numpy()                 # mesmo subconjunto da curva do v2
val_blend, _, _ = sample_dataframe(val_70, VAL_BLEND_SIZE, stratify_col='_y_bin', seed=SEED)
y_vb = val_blend[TARGET].to_numpy()
te_probe = te_eval.iloc[:BATCH_SIZE]
print(f'Dados em {time.time() - t0:.1f}s | treino={len(tr_70):,} | val={len(val_70):,} | teste={len(te_eval):,}')

m_base = compute_metrics(y_te, te_eval[BASELINE])

t0 = time.time()
lgbm_full = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.05, num_leaves=31,
                              random_state=SEED, n_jobs=-1, verbose=-1)
lgbm_full.fit(tr_70[FEAT_COLS], tr_70[TARGET], eval_set=[(val_70[FEAT_COLS], val_70[TARGET])],
              callbacks=[lgb.early_stopping(50, verbose=False)])
t_lgbm_full = time.time() - t0
lgbm_full_te = lgbm_full.predict(te_eval[FEAT_COLS])
lgbm_full_vb = lgbm_full.predict(val_blend[FEAT_COLS])
m_lgbm_full = compute_metrics(y_te, lgbm_full_te)
print(f"App: MAE={m_base['MAE']:.3f} | LightGBM 377k: MAE={m_lgbm_full['MAE']:.3f} R2={m_lgbm_full['R2']:.4f}")

results = [
    metric_row('Meituan App Estimate', 0, m_base, 0.0),
    metric_row(f'LightGBM ({len(tr_70):,} rows)', len(tr_70), m_lgbm_full, t_lgbm_full),
]
# Referência do v2 (mesmo split, mesmas features, teste completo)
V2_REFERENCE = [
    {'model': 'v2: TabPFN (1x100,000, median)', 'train_rows': 100_000, 'test_MAE': 5.073, 'test_RMSE': 6.642, 'test_R2': 0.6841, 'runtime_s': 1361.2},
    {'model': 'v2: Blend TabPFN 100k x0.65 + LightGBM', 'train_rows': 377_104, 'test_MAE': 5.056, 'test_RMSE': 6.590, 'test_R2': 0.6891, 'runtime_s': 1470.0},
]
lc_rows = []   # pontos novos da curva de aprendizado

## Stage 3: Sonda de memória e plano de execução

Para cada `n_estimators` candidato, a sonda faz fit + predict com 10k e 30k linhas e mede o pico
de VRAM. O pico é extrapolado linearmente para 200k/300k e corrigido pelo valor real do v2
(8,71 GB em 100k), caso a extrapolação subestime. O plano escolhe, para cada tamanho, o maior
`n_estimators` cujo pico previsto cabe em `VRAM_SAFETY` × VRAM livre.

Os estimadores internos são um ensemble de pré-processamentos sobre o **mesmo** contexto.
Menos estimadores reduzem memória e tempo, mas podem custar um pouco de precisão. O v3 mede esse
efeito: compare o resultado de 200k/300k com o ganho previsto pela curva do v2.

In [ ]:
free_gpu()
vram_free_gb = torch.cuda.mem_get_info()[0] / 1e9
VRAM_BUDGET_GB = VRAM_SAFETY * vram_free_gb
print(f'VRAM livre: {vram_free_gb:.1f} GB -> orçamento para o pico: {VRAM_BUDGET_GB:.1f} GB')

_probe_cache = {}


def probe(n, n_est):
    """Fit + predict on an n-row context; return peak VRAM (GB), fit time and resolved n_estimators."""
    key = (n, n_est)
    if key in _probe_cache:
        return _probe_cache[key]
    sub, _, _ = sample_dataframe(tr_70, n, stratify_col='_y_bin', seed=SEED)
    free_gpu()
    torch.cuda.reset_peak_memory_stats()
    out, oom = None, False
    try:
        t0 = time.time()
        m = TabPFNModel(make_cfg(n, n_est)).fit(sub, sub[TARGET], num_cols=FEAT_COLS, stratify_col=None)
        fit_s = time.time() - t0
        m.predict(te_probe, output_type=OUTPUT_TYPE)
        out = {'peak_gb': torch.cuda.max_memory_allocated() / 1e9, 'fit_s': fit_s,
               'resolved': resolved_n_estimators(m)}
        del m
    except Exception as exc:
        if not is_oom(exc):
            raise
        oom = True
    free_gpu()
    if oom:
        out = {'peak_gb': float('inf'), 'fit_s': float('nan'), 'resolved': None}
    _probe_cache[key] = out
    return out


def predicted_peak_gb(n_est, n):
    n1, n2 = PROBE_SIZES
    p1, p2 = probe(n1, n_est), probe(n2, n_est)
    slope = (p2['peak_gb'] - p1['peak_gb']) / (n2 - n1)
    return p1['peak_gb'] + slope * (n - n1)


def estimated_fit_s(n_est, n):
    return probe(PROBE_SIZES[1], n_est)['fit_s'] * (n / PROBE_SIZES[1]) ** TIME_EXPONENT


# Calibração com o pico real do v2 (100k, 'auto'): se a extrapolação subestimar, corrige para cima.
pred_100k_auto = predicted_peak_gb('auto', 100_000)
CALIB = max(1.0, V2_PEAK_100K_GB / pred_100k_auto)
print(f"Sonda 'auto': pico previsto em 100k = {pred_100k_auto:.2f} GB (v2 mediu {V2_PEAK_100K_GB} GB) "
      f"-> fator de calibração {CALIB:.2f}")

plan_rows, PLAN = [], {}
for n in TARGET_SIZES:
    if n in FORCE_N_ESTIMATORS:
        PLAN[n] = FORCE_N_ESTIMATORS[n]
        plan_rows.append({'n_train': n, 'n_estimators': PLAN[n], 'note': 'forçado'})
        continue
    PLAN[n] = None
    for n_est in N_EST_CANDIDATES:
        peak = predicted_peak_gb(n_est, n) * CALIB
        fits = peak <= VRAM_BUDGET_GB
        plan_rows.append({
            'n_train': n, 'n_estimators': n_est,
            'resolved_n_estimators': probe(PROBE_SIZES[1], n_est)['resolved'],
            'pred_peak_gb': round(peak, 2), 'fits': fits,
            'est_fit_min': round(estimated_fit_s(n_est, n) / 60, 1),
        })
        if fits:
            PLAN[n] = n_est
            break

plan_df = pd.DataFrame(plan_rows)
plan_df.to_csv(CKPT_DIR / 'colab_v3_memory_plan.csv', index=False)
display(plan_df)
for n, n_est in PLAN.items():
    print(f'  n={n:,}: ' + (f'n_estimators={n_est}' if n_est is not None else 'NÃO cabe em contexto único'))
print('Tempo total (fit + predições) ~ 1,5x o tempo de fit estimado.')

## Stage 4: TabPFN com contexto único de 200k e 300k

Para cada tamanho: fit com o `n_estimators` do plano (se der OOM no fit, tenta o próximo menor),
predições no teste completo (mediana e média) e no subconjunto da validação, LightGBM com a mesma
amostra e blend com peso escolhido na validação. Os resultados são salvos após cada tamanho.

In [ ]:
WEIGHTS = np.round(np.arange(0.0, 1.0001, 0.05), 2)


def best_blend(tab_vb, tab_te):
    maes = [compute_metrics(y_vb, w * tab_vb + (1 - w) * lgbm_full_vb)['MAE'] for w in WEIGHTS]
    w = float(WEIGHTS[int(np.argmin(maes))])
    return w, compute_metrics(y_te, w * tab_te + (1 - w) * lgbm_full_te)


def lgbm_parity(sub):
    t0 = time.time()
    lg = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.05, num_leaves=31,
                           random_state=SEED, n_jobs=-1, verbose=-1)
    lg.fit(sub[FEAT_COLS], sub[TARGET], eval_set=[(val_70[FEAT_COLS], val_70[TARGET])],
           callbacks=[lgb.early_stopping(50, verbose=False)])
    return lg.predict(te_eval[FEAT_COLS]), time.time() - t0


def save_results():
    out = CKPT_DIR / 'colab_v3_results.csv'
    pd.DataFrame(results).to_csv(out, index=False)
    lc_out = CKPT_DIR / 'colab_v3_learning_curve_new_points.csv'
    pd.DataFrame(lc_rows).to_csv(lc_out, index=False)
    backup(out, lc_out, CKPT_DIR / 'colab_v3_memory_plan.csv')


def run_single_context(n, start_n_est):
    candidates = N_EST_CANDIDATES[N_EST_CANDIDATES.index(start_n_est):] \
        if start_n_est in N_EST_CANDIDATES else [start_n_est]
    for n_est in candidates:
        tag = f'colab_v3_70_30_ref_{n}_est{n_est}'
        print(f'\n--- TabPFN contexto único n={n:,} | n_estimators={n_est} ---')
        free_gpu()
        torch.cuda.reset_peak_memory_stats()
        t0 = time.time()
        model, oom = None, False
        try:
            model = TabPFNModel(make_cfg(n, n_est, verbose=True)).fit(
                tr_70, tr_70[TARGET], num_cols=FEAT_COLS, stratify_col='_y_bin',
                log_path=CKPT_DIR / f'{tag}_log.csv')
        except Exception as exc:
            if not is_oom(exc):
                raise
            oom = True
        if oom:
            model = None
            free_gpu()
            print(f'  [OOM no fit] n_estimators={n_est} não coube; tentando o próximo.')
            continue

        fit_s = time.time() - t0
        resolved = resolved_n_estimators(model)
        p_med = predict_with_retry(model, te_eval, 'median')
        p_mean = predict_with_retry(model, te_eval, 'mean') if COMPUTE_MEAN else None
        p_vb = predict_with_retry(model, val_blend, 'median')
        total_s = time.time() - t0
        peak_gb = torch.cuda.max_memory_allocated() / 1e9

        m_med = compute_metrics(y_te, p_med)
        m_mean = compute_metrics(y_te, p_mean) if p_mean is not None else None
        model.record_eval_target(y_te)
        model.save_audit_manifest(CKPT_DIR / f'{tag}_audit.json', extra={
            'split': '70_30', 'stage': 'complete_dispatch_v3', 'feature_set': FEATURE_SET_NAME,
            'sample_size': n, 'n_estimators': n_est, 'resolved_n_estimators': resolved,
            'test_metrics_median': m_med, 'test_metrics_mean': m_mean,
            'fit_s': round(fit_s, 1), 'runtime_s': round(total_s, 1), 'peak_vram_gb': round(peak_gb, 2),
            'batch_size_used': model.config.batch_size,
        })
        pred_df = pd.DataFrame({'y_true': y_te, 'y_pred_median': p_med})
        if p_mean is not None:
            pred_df['y_pred_mean'] = p_mean
        pred_df.to_parquet(CKPT_DIR / f'{tag}_predictions.parquet', index=False)
        del model
        free_gpu()
        print(f'  fit={fit_s/60:.1f} min | total={total_s/60:.1f} min | pico VRAM={peak_gb:.2f} GB | '
              f'n_estimators resolvido={resolved} | MAE={m_med["MAE"]:.3f} R2={m_med["R2"]:.4f}')
        backup(CKPT_DIR / f'{tag}_audit.json', CKPT_DIR / f'{tag}_predictions.parquet')
        return {'n': n, 'n_est': n_est, 'resolved': resolved, 'p_med': p_med, 'p_mean': p_mean,
                'p_vb': p_vb, 'm_med': m_med, 'm_mean': m_mean, 'total_s': total_s, 'peak_gb': peak_gb}
    return None


def record_scenario(label, n, r):
    """Append test metrics, LightGBM parity, blend and learning-curve points for one TabPFN result."""
    extra = {'n_estimators': r['n_est'], 'peak_vram_gb': round(r['peak_gb'], 2)}
    results.append(metric_row(f'{label}, median', n, r['m_med'], r['total_s'], **extra))
    if r['m_mean'] is not None:
        results.append(metric_row(f'{label}, mean', n, r['m_mean'], r['total_s'], **extra))
    sub, _, _ = sample_dataframe(tr_70, n, stratify_col='_y_bin', seed=SEED)
    lg_te, lg_s = lgbm_parity(sub)
    results.append(metric_row(f'LightGBM ({n:,} rows)', n, compute_metrics(y_te, lg_te), lg_s))
    w, m_blend = best_blend(r['p_vb'], r['p_med'])
    results.append(metric_row(f'Blend {label} x{w:.2f} + LightGBM 377k', len(tr_70), m_blend,
                              r['total_s'] + t_lgbm_full, n_estimators=r['n_est']))
    m_tab_lc = compute_metrics(y_lc, r['p_med'][lc_idx])
    m_lg_lc = compute_metrics(y_lc, lg_te[lc_idx])
    lc_rows.append({'n_train': n, 'model': 'TabPFN-v3.5', 'variant': label, 'MAE': round(m_tab_lc['MAE'], 3),
                    'RMSE': round(m_tab_lc['RMSE'], 3), 'R2': round(m_tab_lc['R2'], 4)})
    lc_rows.append({'n_train': n, 'model': 'LightGBM', 'variant': 'LightGBM', 'MAE': round(m_lg_lc['MAE'], 3),
                    'RMSE': round(m_lg_lc['RMSE'], 3), 'R2': round(m_lg_lc['R2'], 4)})
    save_results()
    print(f'  {label}: MAE={r["m_med"]["MAE"]:.3f} | blend w={w:.2f} MAE={m_blend["MAE"]:.3f} R2={m_blend["R2"]:.4f}')


single_runs = {}
for n in TARGET_SIZES:
    if PLAN[n] is None:
        print(f'\nn={n:,} não cabe em contexto único na {gpu_name}; ver Stage 5.')
        single_runs[n] = None
        continue
    r = run_single_context(n, PLAN[n])
    single_runs[n] = r
    if r is not None:
        record_scenario(f'TabPFN (1x{n:,}, est={r["n_est"]})', n, r)

display(pd.DataFrame(results))

## Stage 5: Fallback para 300k — 3 contextos disjuntos de 100k

Roda quando 300k não coube em contexto único (`RUN_DISJOINT_ENSEMBLE='auto'`) ou sempre, com `True`.
A amostra estratificada de 300k é dividida em 3 partes disjuntas de 100k. Cada parte é um modelo
com a configuração do v2 (que sabidamente cabe na T4), e as predições são somadas e divididas por 3.
Só um modelo fica na GPU por vez.

Diferença para o contexto único: cada predição só "enxerga" 100k linhas por vez. A média reduz
variância, mas não dá ao modelo o contexto maior. Por isso o contexto único é preferível quando cabe.

In [ ]:
n_total = max(TARGET_SIZES)
need_fallback = single_runs.get(n_total) is None
if RUN_DISJOINT_ENSEMBLE is True or (RUN_DISJOINT_ENSEMBLE == 'auto' and need_fallback):
    sub_all, _, _ = sample_dataframe(tr_70, n_total, stratify_col='_y_bin', seed=SEED)
    perm = np.random.default_rng(SEED).permutation(len(sub_all))
    parts = np.array_split(perm, math.ceil(len(sub_all) / DISJOINT_MEMBER_SIZE))
    acc = {'p_med': 0.0, 'p_mean': 0.0, 'p_vb': 0.0}
    t_start, peak = time.time(), 0.0
    for k, part in enumerate(parts, 1):
        part_df = sub_all.iloc[np.sort(part)]
        print(f'\n--- Membro disjunto {k}/{len(parts)} (n={len(part_df):,}) ---')
        free_gpu()
        torch.cuda.reset_peak_memory_stats()
        m = TabPFNModel(make_cfg(len(part_df), 'auto', verbose=True)).fit(
            part_df, part_df[TARGET], num_cols=FEAT_COLS, stratify_col=None)
        acc['p_med'] = acc['p_med'] + predict_with_retry(m, te_eval, 'median')
        if COMPUTE_MEAN:
            acc['p_mean'] = acc['p_mean'] + predict_with_retry(m, te_eval, 'mean')
        acc['p_vb'] = acc['p_vb'] + predict_with_retry(m, val_blend, 'median')
        peak = max(peak, torch.cuda.max_memory_allocated() / 1e9)
        del m
        free_gpu()
        print(f'  acumulado: {(time.time() - t_start) / 60:.1f} min')
    p_med = acc['p_med'] / len(parts)
    p_mean = acc['p_mean'] / len(parts) if COMPUTE_MEAN else None
    r = {'n_est': 'auto', 'p_med': p_med, 'p_mean': p_mean, 'p_vb': acc['p_vb'] / len(parts),
         'm_med': compute_metrics(y_te, p_med),
         'm_mean': compute_metrics(y_te, p_mean) if p_mean is not None else None,
         'total_s': time.time() - t_start, 'peak_gb': peak}
    label = f'TabPFN ({len(parts)}x{DISJOINT_MEMBER_SIZE:,} disjuntos)'
    out = CKPT_DIR / f'colab_v3_70_30_ref_{n_total}_disjoint_predictions.parquet'
    pd.DataFrame({'y_true': y_te, 'y_pred_median': p_med,
                  **({'y_pred_mean': p_mean} if p_mean is not None else {})}).to_parquet(out, index=False)
    backup(out)
    record_scenario(label, n_total, r)
    display(pd.DataFrame(results))
else:
    print('Fallback não necessário: 300k rodou em contexto único (ou RUN_DISJOINT_ENSEMBLE=False).')

## Stage 6: Resultados, curva de aprendizado completa (v2 + v3) e artefatos

A curva junta os pontos do v2 (10k a 100k, `n_estimators='auto'`) com os novos pontos do v3,
todos no **mesmo** subconjunto de 10.000 linhas do teste.

In [ ]:
V2_LC = pd.DataFrame([
    (10_000, 'TabPFN-v3.5', 5.319, 6.986, 0.6508), (10_000, 'LightGBM', 5.509, 7.157, 0.6335),
    (25_000, 'TabPFN-v3.5', 5.232, 6.871, 0.6622), (25_000, 'LightGBM', 5.397, 7.006, 0.6489),
    (50_000, 'TabPFN-v3.5', 5.156, 6.779, 0.6713), (50_000, 'LightGBM', 5.305, 6.896, 0.6598),
    (75_000, 'TabPFN-v3.5', 5.107, 6.722, 0.6767), (75_000, 'LightGBM', 5.282, 6.873, 0.6620),
    (100_000, 'TabPFN-v3.5', 5.089, 6.692, 0.6796), (100_000, 'LightGBM', 5.257, 6.840, 0.6652),
], columns=['n_train', 'model', 'MAE', 'RMSE', 'R2'])
V2_LC['variant'] = np.where(V2_LC['model'] == 'LightGBM', 'LightGBM', 'v2 (est=auto)')
V2_LC['source'] = 'v2'
lc_new = pd.DataFrame(lc_rows)
if not lc_new.empty:
    lc_new['source'] = 'v3'
lc_all = pd.concat([V2_LC, lc_new], ignore_index=True)
lc_all.to_csv(CKPT_DIR / 'colab_v3_learning_curve.csv', index=False)

results_df = pd.concat([pd.DataFrame(results), pd.DataFrame(V2_REFERENCE)], ignore_index=True)
results_df = results_df.sort_values('test_MAE').reset_index(drop=True)
results_df.to_csv(CKPT_DIR / 'colab_v3_results.csv', index=False)
display(results_df)
display(lc_all.sort_values(['model', 'n_train']))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
dfp = results_df[results_df['model'] != 'Meituan App Estimate'].sort_values('test_MAE', ascending=False)
colors = ['#dc2626' if 'Blend' in m else '#0284c7' if 'TabPFN' in m else '#16a34a' for m in dfp['model']]
axes[0].barh(dfp['model'], dfp['test_MAE'], color=colors, edgecolor='white')
axes[0].set_xlim(float(dfp['test_MAE'].min()) - 0.1, float(dfp['test_MAE'].max()) + 0.08)
for i, v in enumerate(dfp['test_MAE']):
    axes[0].text(v + 0.005, i, f'{v:.3f}', va='center', fontsize=8)
axes[0].set_title('MAE no teste completo (70/30)')
axes[0].set_xlabel('MAE (min)')

lg_pts = lc_all[lc_all['model'] == 'LightGBM'].sort_values('n_train')
axes[1].plot(lg_pts['n_train'], lg_pts['MAE'], 's-', color='#16a34a', label='LightGBM (mesma amostra)')
v2_pts = lc_all[(lc_all['model'] == 'TabPFN-v3.5') & (lc_all['source'] == 'v2')].sort_values('n_train')
axes[1].plot(v2_pts['n_train'], v2_pts['MAE'], 'o-', color='#0284c7', label='TabPFN v2 (est=auto)')
if not lc_new.empty:
    for _, row in lc_new[lc_new['model'] == 'TabPFN-v3.5'].iterrows():
        axes[1].scatter(row['n_train'], row['MAE'], s=70, marker='D', color='#7c3aed', zorder=5)
        axes[1].annotate(row['variant'].replace('TabPFN ', ''), (row['n_train'], row['MAE']),
                         textcoords='offset points', xytext=(-10, 8), fontsize=8, ha='right')
# tendência do v2: -0,07 min por duplicação
xs = np.array([10_000, 400_000])
axes[1].plot(xs, 5.085 - 0.0712 * np.log2(xs / 100_000), ':', color='#0284c7', alpha=0.6,
             label='Tendência do v2 (−0,07/duplicação)')
axes[1].set_xscale('log')
axes[1].set_title(f'Curva de aprendizado (subconjunto de {LC_EVAL_SIZE:,} linhas do teste)')
axes[1].set_xlabel('Linhas de treino (escala log)')
axes[1].set_ylabel('MAE (min)')
axes[1].legend(fontsize=8)
plt.tight_layout()
fig_path = CKPT_DIR / 'colab_v3_tabpfn_diagnostics.png'
plt.savefig(fig_path, dpi=160, bbox_inches='tight')
plt.show()

zip_path = CKPT_DIR / 'tabpfn_colab_v3_artifacts.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(CKPT_DIR.glob('colab_v3_*')):
        zf.write(f, arcname=f.name)
backup(zip_path)
print(f'Artefatos: {zip_path} ({zip_path.stat().st_size / 1e6:.2f} MB)')
if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))